# Ingest NutriAccess Food Desert GIS Data

This notebook connects to the project S3 bucket, verifies the raw GIS files exist, loads OpenStreetMap POI shapefiles for Northern and Southern California, filters food retail locations, and saves processed outputs for later analysis.

This notebook is specific to the GIS data and is used as an optional enrichment dataset for the NutriAccess project.

### 5. Grocery Store Location Data (optional enrichment data)

https://download.geofabrik.de/

Source: OpenStreetMap (via Geofabrik download)
Map the geographic distribution of food retailers such as supermarkets, grocery stores,
and convenience stores to analyze spatial access to food retailers

global store location data that can be filtered for:

    - supermarkets
    - grocery stores
    - convenience stores

Purpose in project: Map the geographic distribution of food retailers and analyze
spatial access

Downloaded Geofabrik Shapefiles for Northern and Southern California. 

#### NOTE: Raw Datafiles for this project were downloaded on March 3rd, 2026, at 7:00 am 

## Import Libaries 

In [2]:
import boto3
import pandas as pd
import geopandas as gpd
import os

## Define Project Configuration

In [3]:
# Keep path consistent for the whole team! 
BUCKET = "nutriaccess-data"
RAW_PREFIX = "rawData/"
PROCESSED_PREFIX = "processedData/"

## Connect to S3

In [4]:
s3 = boto3.client("s3")

## List every file that exists in the  raw data folder

*Note: Many files will appear; some interact with each other and remain part of the same dataset. For example, Geospatial datasets may contain many interacting files that together make up a single shapefile dataset..*

In [5]:
# Use paginator in case there are many files
paginator = s3.get_paginator("list_objects_v2")

pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=RAW_PREFIX
)

files = []

for page in pages:
    for obj in page.get("Contents", []):
        files.append(obj["Key"])

print("Files found in rawData:")
for f in files:
    print(f)

Files found in rawData:
rawData/
rawData/ACSDP1Y2024.DP05-2026-03-13T140903.csv
rawData/ACSST1Y2024.S1701-2026-03-13T140807.csv
rawData/ACSST1Y2024.S1901-2026-03-13T140835.csv
rawData/FoodAccess/
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/ReadMe.csv
rawData/FoodAccess/2019_Food_Access_Research_Atlas_Data/VariableLookup.csv
rawData/FoodEnvironment/
rawData/FoodEnvironment/2025-food-environment-atlas-data/ReadMeFile2025.txt
rawData/FoodEnvironment/2025-food-environment-atlas-data/StateAndCountyData.csv
rawData/FoodEnvironment/2025-food-environment-atlas-data/VariableList.csv
rawData/PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260313.csv
rawData/geofabrik_NorCal/
rawData/geofabrik_NorCal/norcal-260312-free.shp/README
rawData/geofabrik_NorCal/norcal-260312-free.shp/gis_osm_buildings_a_free_1.cpg
rawData/geofabrik_NorCal/norcal-260312-free.shp/gis_osm_buildings_a_free_1.

## Load GIS files 

In [6]:
pois_socal = gpd.read_file(
    f"s3://{BUCKET}/rawData/geofabrik_SoCal/socal-260312-free.shp/gis_osm_pois_free_1.shp"
)

pois_socal.head()

,osm_id,code,fclass,name,geometry
0,49678192,2907,camera_surveillance,None,POINT (-117.30826 33.08287)
1,49736461,2742,viewpoint,Inspiration Point,POINT (-116.56872 33.05023)
2,54150679,2701,tourist_info,None,POINT (-117.11529 34.01412)
3,54159406,2907,camera_surveillance,None,POINT (-117.50287 33.76408)
4,54208222,2741,picnic_site,Lake Fulmor Picnic Area,POINT (-116.78137 33.80449)


In [7]:
pois_norcal = gpd.read_file(
    f"s3://{BUCKET}/rawData/geofabrik_NorCal/norcal-260312-free.shp/gis_osm_pois_free_1.shp"
)

pois_norcal.head()

,osm_id,code,fclass,name,geometry
0,15972546,2601,bank,Chase,POINT (-122.02073 36.97937)
1,26637661,2742,viewpoint,Tunnel View,POINT (-119.67689 37.71557)
2,29946571,2901,toilet,None,POINT (-121.41902 40.68548)
3,33112985,2721,attraction,Captain Jack's Stronghold,POINT (-121.50175 41.82345)
4,33113436,2721,attraction,Fleener Chimneys,POINT (-121.56817 41.75818)


## Combine GIS files 

In [8]:
pois_ca = pd.concat([pois_norcal, pois_socal], ignore_index=True)
print(pois_ca.shape)

(224390, 5)


# Inspect the columns 

In [9]:
print(pois_ca.columns.tolist())

['osm_id', 'code', 'fclass', 'name', 'geometry']


## Filter for Store Type 

In [10]:
store_types = ["supermarket", "convenience", "grocery"]

filtered_stores = pois_ca[
    pois_ca["fclass"].str.lower().isin(store_types)
].copy()

print(filtered_stores[["name", "fclass"]].head())
print(filtered_stores["fclass"].value_counts())

                   name       fclass
144  California Oranges  supermarket
168        Trader Joe's  supermarket
221           Lunardi's  supermarket
238  India Cash & Carry  supermarket
240     Kumud Groceries  supermarket
fclass
convenience    3383
supermarket    2205
Name: count, dtype: int64


## Inspect Unique Classes 

In [11]:
sorted(pois_ca["fclass"].dropna().str.lower().unique())[:100]

['alpine_hut',
 'archaeological',
 'arts_centre',
 'artwork',
 'atm',
 'attraction',
 'bakery',
 'bank',
 'bar',
 'battlefield',
 'beauty_shop',
 'bench',
 'beverages',
 'bicycle_rental',
 'bicycle_shop',
 'biergarten',
 'bookshop',
 'butcher',
 'cafe',
 'camera_surveillance',
 'camp_site',
 'car_dealership',
 'car_rental',
 'car_sharing',
 'car_wash',
 'caravan_site',
 'castle',
 'chalet',
 'chemist',
 'cinema',
 'clinic',
 'clothes',
 'college',
 'comms_tower',
 'community_centre',
 'computer_shop',
 'convenience',
 'courthouse',
 'dentist',
 'department_store',
 'doctors',
 'dog_park',
 'doityourself',
 'drinking_water',
 'embassy',
 'fast_food',
 'fire_station',
 'florist',
 'food_court',
 'fort',
 'fountain',
 'furniture_shop',
 'garden_centre',
 'general',
 'gift_shop',
 'golf_course',
 'graveyard',
 'greengrocer',
 'guesthouse',
 'hairdresser',
 'hospital',
 'hostel',
 'hotel',
 'hunting_stand',
 'ice_rink',
 'jeweller',
 'kindergarten',
 'kiosk',
 'laundry',
 'library',
 'light

In [12]:
# Check for supermarket 
sorted(pois_ca["fclass"].unique())

['alpine_hut',
 'archaeological',
 'arts_centre',
 'artwork',
 'atm',
 'attraction',
 'bakery',
 'bank',
 'bar',
 'battlefield',
 'beauty_shop',
 'bench',
 'beverages',
 'bicycle_rental',
 'bicycle_shop',
 'biergarten',
 'bookshop',
 'butcher',
 'cafe',
 'camera_surveillance',
 'camp_site',
 'car_dealership',
 'car_rental',
 'car_sharing',
 'car_wash',
 'caravan_site',
 'castle',
 'chalet',
 'chemist',
 'cinema',
 'clinic',
 'clothes',
 'college',
 'comms_tower',
 'community_centre',
 'computer_shop',
 'convenience',
 'courthouse',
 'dentist',
 'department_store',
 'doctors',
 'dog_park',
 'doityourself',
 'drinking_water',
 'embassy',
 'fast_food',
 'fire_station',
 'florist',
 'food_court',
 'fort',
 'fountain',
 'furniture_shop',
 'garden_centre',
 'general',
 'gift_shop',
 'golf_course',
 'graveyard',
 'greengrocer',
 'guesthouse',
 'hairdresser',
 'hospital',
 'hostel',
 'hotel',
 'hunting_stand',
 'ice_rink',
 'jeweller',
 'kindergarten',
 'kiosk',
 'laundry',
 'library',
 'light

# Filter Data for Project 

In [13]:
store_classes = [
    "supermarket",
    "convenience",
    "greengrocer",
    "general",
    "department_store",
    "market_place"
]

food_stores = pois_ca[
    pois_ca["fclass"].isin(store_classes)
].copy()

print(food_stores["fclass"].value_counts())
print(food_stores.head())

fclass
convenience         3383
supermarket         2205
department_store     756
market_place         191
greengrocer          164
general               91
Name: count, dtype: int64
        osm_id  code            fclass                name  \
144  150943454  2501       supermarket  California Oranges   
168  212435197  2501       supermarket        Trader Joe's   
221  262936489  2501       supermarket           Lunardi's   
236  266625620  2505  department_store      dd's Discounts   
238  266630158  2501       supermarket  India Cash & Carry   

                        geometry  
144  POINT (-118.82807 36.10124)  
168  POINT (-122.25261 37.84554)  
221  POINT (-122.38498 37.59454)  
236  POINT (-122.02629 37.36126)  
238   POINT (-122.00569 37.3514)  


## Check how many store were found 

In [14]:
print("Total stores:", food_stores.shape[0])

Total stores: 6790


In [15]:
food_stores["fclass"].value_counts()

fclass
convenience         3383
supermarket         2205
department_store     756
market_place         191
greengrocer          164
general               91
Name: count, dtype: int64

# Save your filtered dataset

In [16]:
# Spatial Dataset 
food_stores.to_file(
    "processed/california_food_store_locations.geojson",
    driver="GeoJSON"
)

# Tabular Version 
food_stores.to_csv("processed/california_food_store_locations.csv", index=False)


# Save filtered Dataset to S3 Bucket 

In [17]:
s3.upload_file(
    "processed/california_food_store_locations.csv",
    BUCKET,
    "processedData/california_food_store_locations.csv"
)

s3.upload_file(
    "processed/california_food_store_locations.geojson",
    BUCKET,
    "processedData/california_food_store_locations.geojson"
)

# Release Resources

In [18]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [19]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>